In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install torch pandas numpy scikit-learn --quiet --break-system-packages

import os
import ast
import time
import random
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

# ---- Shared signal contract — must match Week 1 prep + ECGDataset notebooks ----
TARGET_FS = 500
TARGET_SECONDS = 10
TARGET_LEN = TARGET_FS * TARGET_SECONDS   # 5000
NUM_LEADS = 12
SEED = 42

CLASS_NAMES = ['NSR', 'AFIB_AFL', 'IAVB', 'LBBB', 'RBBB']
NUM_CLASSES = len(CLASS_NAMES)

DRIVE_ROOT = '/content/drive/MyDrive/LSTS/ecg_benchmark'
PROCESSED_DIR = f'{DRIVE_ROOT}/processed'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'
LOG_DIR = f'{DRIVE_ROOT}/logs'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", DEVICE)

ALL_DATASETS = ['ptbxl', 'cpsc2018', 'georgia', 'mimic_iv', 'code_ii']


Mounted at /content/drive
Using device: cpu


In [2]:
def set_seed(seed=SEED):
    """
    Called at the start of every training run. Full determinism
    (cudnn.deterministic=True) trades some GPU speed for
    reproducibility — worth it here since the team needs runs to
    be comparable across people, not just fast.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
def parse_raw_labels(raw_labels_str, dataset):
    if pd.isna(raw_labels_str) or raw_labels_str == '':
        return []
    if dataset == 'ptbxl':
        try:
            return list(ast.literal_eval(raw_labels_str).keys())
        except (ValueError, SyntaxError):
            return []
    elif dataset in ('cpsc2018', 'georgia', 'mimic_iv', 'code_ii'):
        return [c.strip() for c in str(raw_labels_str).split(',') if c.strip()]
    else:
        raise ValueError(f"Unknown dataset for label parsing: {dataset}")


def build_label_vector(raw_labels_str, dataset, mapping_df, nsr_variant='A'):
    codes = parse_raw_labels(raw_labels_str, dataset)
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    matched_any = False
    subset = mapping_df[mapping_df['dataset'] == dataset]
    for code in codes:
        rows = subset[subset['raw_code'] == code]
        if rows.empty:
            continue
        matched_any = True
        for _, row in rows.iterrows():
            if row['mapped_class'] == 'EXCLUDE':
                continue
            if row['mapped_class'] == 'NSR' and row['nsr_variant'] not in (nsr_variant, 'NA'):
                continue
            vec[CLASS_NAMES.index(row['mapped_class'])] = 1.0
    return vec, ('ok' if matched_any else 'no_match')


def assign_splits(index_df, seed=SEED, val_frac=0.1, test_frac=0.2):
    def split_for(ecg_id):
        h = int(hashlib.md5(f"{seed}_{ecg_id}".encode()).hexdigest(), 16)
        frac = (h % 10000) / 10000.0
        if frac < test_frac:
            return 'test'
        elif frac < test_frac + val_frac:
            return 'val'
        return 'train'
    index_df = index_df.copy()
    index_df['split'] = index_df['ecg_id'].astype(str).apply(split_for)
    return index_df


class ECGDataset(Dataset):
    def __init__(self, dataset_names, split, index_dir, mapping_df,
                 nsr_variant='A', drop_no_match=True, verbose=True):
        self.index_dir = index_dir
        rows = []
        no_match_count = 0
        for dataset in dataset_names:
            index_path = f"{index_dir}/{dataset}_index.csv"
            if not os.path.exists(index_path):
                raise FileNotFoundError(f"No index CSV for '{dataset}' at {index_path}")
            df = pd.read_csv(index_path)
            if 'error' in df.columns:
                df = df[df['error'].isna()]
            if 'shape_ok' in df.columns:
                df = df[df['shape_ok'] == True]
            df = assign_splits(df)
            df = df[df['split'] == split]
            for _, row in df.iterrows():
                label_vec, status = build_label_vector(row.get('raw_labels'), dataset, mapping_df, nsr_variant)
                if status == 'no_match':
                    no_match_count += 1
                    if drop_no_match:
                        continue
                rows.append({'ecg_id': row['ecg_id'], 'dataset': dataset, 'label': label_vec})
        self.samples = rows
        if verbose:
            print(f"[ECGDataset] {split}, datasets={dataset_names}: {len(self.samples)} recordings "
                  f"({no_match_count} unmatched {'dropped' if drop_no_match else 'kept'})")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        file_path = f"{self.index_dir}/{s['dataset']}/signals/{s['ecg_id']}.npy"
        sig = np.load(file_path)
        return torch.from_numpy(sig).float(), torch.from_numpy(s['label']).float()


In [4]:
def compute_auroc(y_true, y_probs):
    """
    y_true, y_probs: numpy arrays of shape (N, NUM_CLASSES)

    Returns (macro_auroc, per_class_auroc_dict).

    A class with zero positives (or zero negatives) in this split has
    an undefined AUROC — sklearn would raise an error. We skip that
    class from the macro average and report it as NaN rather than
    crashing or silently substituting a fake value, since CPSC2018's
    small test set makes this a real possibility, not an edge case.
    """
    per_class = {}
    valid_scores = []
    for i, cls in enumerate(CLASS_NAMES):
        y_t = y_true[:, i]
        y_p = y_probs[:, i]
        if len(np.unique(y_t)) < 2:
            per_class[cls] = float('nan')
            continue
        score = roc_auc_score(y_t, y_p)
        per_class[cls] = score
        valid_scores.append(score)

    macro_auroc = float(np.mean(valid_scores)) if valid_scores else float('nan')
    return macro_auroc, per_class


In [5]:
class EarlyStopper:
    """
    Tracks validation macro-AUROC across epochs. Call .step(score, model)
    after each validation pass. Saves the best-so-far model weights to
    Drive automatically — you don't need to manually save checkpoints
    in your training loop.
    """
    def __init__(self, checkpoint_path, patience=10, mode='max'):
        self.checkpoint_path = checkpoint_path
        self.patience = patience
        self.mode = mode
        self.best_score = -float('inf') if mode == 'max' else float('inf')
        self.counter = 0
        self.should_stop = False

    def step(self, score, model):
        improved = (score > self.best_score) if self.mode == 'max' else (score < self.best_score)
        if improved:
            self.best_score = score
            self.counter = 0
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return improved

In [6]:
def train_model(model, train_loader, val_loader, run_name,
                 num_epochs=50, lr=1e-3, patience=10, seed=SEED,
                 checkpoint_dir=CHECKPOINT_DIR, log_dir=LOG_DIR):
    set_seed(seed)
    model = model.to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    use_amp = (DEVICE.type == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    checkpoint_path = f"{checkpoint_dir}/{run_name}_best.pt"
    log_path = f"{log_dir}/{run_name}_log.csv"
    stopper = EarlyStopper(checkpoint_path, patience=patience, mode='max')

    log_rows = []

    for epoch in range(num_epochs):
        # ---- Train ----
        model.train()
        train_loss_total = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss_total += loss.item() * x.size(0)

        train_loss = train_loss_total / len(train_loader.dataset)

        # ---- Validate ----
        model.eval()
        val_loss_total = 0.0
        all_probs, all_labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    logits = model(x)
                    loss = criterion(logits, y)
                val_loss_total += loss.item() * x.size(0)
                all_probs.append(torch.sigmoid(logits).cpu().numpy())
                all_labels.append(y.cpu().numpy())

        val_loss = val_loss_total / len(val_loader.dataset)
        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        val_macro_auroc, val_per_class = compute_auroc(all_labels, all_probs)

        improved = stopper.step(val_macro_auroc, model)

        row = {
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_macro_auroc': val_macro_auroc,
            **{f'val_auroc_{cls}': val_per_class[cls] for cls in CLASS_NAMES},
            'improved': improved,
        }
        log_rows.append(row)
        pd.DataFrame(log_rows).to_csv(log_path, index=False)  # rewritten each epoch — safe if Colab disconnects

        print(f"[{run_name}] epoch {epoch}: train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} val_macro_auroc={val_macro_auroc:.4f}"
              f"{'  (best so far, checkpointed)' if improved else ''}")

        if stopper.should_stop:
            print(f"[{run_name}] Early stopping at epoch {epoch} "
                  f"(no improvement for {patience} epochs). "
                  f"Best val macro-AUROC: {stopper.best_score:.4f}")
            break

    # Reload best checkpoint before returning, so the caller always
    # has the best-performing weights, not just whatever the last epoch left.
    model.load_state_dict(torch.load(checkpoint_path))
    return {'log_path': log_path, 'checkpoint_path': checkpoint_path,
            'best_val_macro_auroc': stopper.best_score}


In [7]:
def evaluate_model(model, test_loader):
    """
    Run a trained model against a test DataLoader and return
    (macro_auroc, per_class_auroc_dict). Use this for the
    in-distribution test-set numbers Week 2 requires, and again in
    Week 3 for every off-diagonal cross-dataset cell.
    """
    model = model.to(DEVICE)
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            logits = model(x)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(y.numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    return compute_auroc(all_labels, all_probs)


# %% CELL 8 — InceptionTime architecture (Siddharth's assigned model)
class InceptionModule(nn.Module):
    """One Inception module: a bottleneck, three parallel convolutions
    at different kernel sizes, and a max-pool branch, concatenated."""
    def __init__(self, in_channels, out_channels=32, bottleneck_channels=32,
                 kernel_sizes=(9, 19, 39)):
        super().__init__()
        self.use_bottleneck = in_channels > 1
        bn_channels = bottleneck_channels if self.use_bottleneck else in_channels

        self.bottleneck = (nn.Conv1d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
                            if self.use_bottleneck else nn.Identity())

        self.convs = nn.ModuleList([
            nn.Conv1d(bn_channels, out_channels, kernel_size=k, padding=k // 2, bias=False)
            for k in kernel_sizes
        ])

        self.pool = nn.MaxPool1d(kernel_size=3, stride=1, padding=1)
        self.pool_conv = nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)

        total_channels = out_channels * (len(kernel_sizes) + 1)
        self.bn = nn.BatchNorm1d(total_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        bottleneck_out = self.bottleneck(x)
        conv_outs = [conv(bottleneck_out) for conv in self.convs]
        pool_out = self.pool_conv(self.pool(x))
        out = torch.cat(conv_outs + [pool_out], dim=1)
        return self.relu(self.bn(out))


class InceptionTime(nn.Module):
    """
    Standard InceptionTime for multivariate time series classification
    (Fawaz et al., 2019), adapted for 12-lead ECG input and 5-class
    multi-label output. 6 Inception modules with a residual connection
    every 3 modules, global average pooling, then a linear classifier
    head. Returns raw logits — use BCEWithLogitsLoss / sigmoid outside.
    """
    def __init__(self, in_channels=NUM_LEADS, num_classes=NUM_CLASSES,
                 num_modules=6, out_channels=32, bottleneck_channels=32):
        super().__init__()
        self.modules_list = nn.ModuleList()
        self.residual_convs = nn.ModuleList()
        self.residual_bns = nn.ModuleList()

        channels = in_channels
        module_out_channels = out_channels * 4  # 3 convs + pool branch

        for i in range(num_modules):
            self.modules_list.append(
                InceptionModule(channels, out_channels, bottleneck_channels)
            )
            channels = module_out_channels

            if (i + 1) % 3 == 0:
                self.residual_convs.append(
                    nn.Conv1d(in_channels if i < 3 else module_out_channels,
                              module_out_channels, kernel_size=1, bias=False)
                )
                self.residual_bns.append(nn.BatchNorm1d(module_out_channels))

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(module_out_channels, num_classes)

    def forward(self, x):
        residual_input = x
        residual_idx = 0

        for i, module in enumerate(self.modules_list):
            x = module(x)
            if (i + 1) % 3 == 0:
                res = self.residual_convs[residual_idx](residual_input)
                res = self.residual_bns[residual_idx](res)
                x = torch.relu(x + res)
                residual_input = x
                residual_idx += 1

        x = self.gap(x).squeeze(-1)
        return self.fc(x)


In [ ]:
mock_mapping_rows = [
    ('ptbxl',    'SR',         'NSR',        'MOCK-A',       'A'),
    ('ptbxl',    'NORM',       'NSR',        'MOCK-B',       'B'),
    ('ptbxl',    'AFIB',       'AFIB_AFL',   'MOCK',         'NA'),
    ('ptbxl',    '1AVB',       'IAVB',       'MOCK',         'NA'),
    ('cpsc2018', 'Normal',     'NSR',        'MOCK-B',       'B'),
    ('cpsc2018', 'AF',         'AFIB_AFL',   'MOCK',         'NA'),
    ('cpsc2018', 'I-AVB',      'IAVB',       'MOCK',         'NA'),
]
mapping_df = pd.DataFrame(
    mock_mapping_rows, columns=['dataset', 'raw_code', 'mapped_class', 'rule_id', 'nsr_variant']
)

results = []

for dataset in ALL_DATASETS:
    print(f"\n{'='*60}\nTraining InceptionTime on {dataset}\n{'='*60}")

    train_ds = ECGDataset([dataset], 'train', PROCESSED_DIR, mapping_df, verbose=True)
    val_ds = ECGDataset([dataset], 'val', PROCESSED_DIR, mapping_df, verbose=True)
    test_ds = ECGDataset([dataset], 'test', PROCESSED_DIR, mapping_df, verbose=True)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

    model = InceptionTime()
    train_result = train_model(
        model, train_loader, val_loader,
        run_name=f'inceptiontime_{dataset}',
        num_epochs=50, lr=1e-3, patience=10,
    )

    test_macro_auroc, test_per_class = evaluate_model(model, test_loader)

    print(f"[{dataset}] In-distribution TEST macro-AUROC: {test_macro_auroc:.4f}")
    print(f"[{dataset}] Per-class: {test_per_class}")

    results.append({
        'architecture': 'InceptionTime',
        'dataset': dataset,
        'test_macro_auroc': test_macro_auroc,
        **{f'test_auroc_{cls}': test_per_class[cls] for cls in CLASS_NAMES},
        'best_val_macro_auroc': train_result['best_val_macro_auroc'],
        'checkpoint_path': train_result['checkpoint_path'],
    })

results_df = pd.DataFrame(results)
results_path = f'{PROCESSED_DIR}/inceptiontime_indistribution_results.csv'
results_df.to_csv(results_path, index=False)
print(f"\nSaved results to {results_path}")



Training InceptionTime on ptbxl
[ECGDataset] train, datasets=['ptbxl']: 13803 recordings (1473 unmatched dropped)
[ECGDataset] val, datasets=['ptbxl']: 1946 recordings (207 unmatched dropped)
[ECGDataset] test, datasets=['ptbxl']: 3959 recordings (411 unmatched dropped)


/tmp/ipykernel_5854/2965848194.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipykernel_5854/2965848194.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_5854/2965848194.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[inceptiontime_ptbxl] epoch 0: train_loss=0.1747 val_loss=0.1286 val_macro_auroc=0.8812  (best so far, checkpointed)


/tmp/ipykernel_5854/2965848194.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_5854/2965848194.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[inceptiontime_ptbxl] epoch 1: train_loss=0.1083 val_loss=0.0945 val_macro_auroc=0.9337  (best so far, checkpointed)


/tmp/ipykernel_5854/2965848194.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_5854/2965848194.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[inceptiontime_ptbxl] epoch 2: train_loss=0.0890 val_loss=0.0822 val_macro_auroc=0.9392  (best so far, checkpointed)


/tmp/ipykernel_5854/2965848194.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


In [1]:
summary_cols = ['dataset', 'test_macro_auroc'] + [f'test_auroc_{cls}' for cls in CLASS_NAMES]
summary = results_df[summary_cols].copy()
summary.columns = ['Dataset', 'Macro-AUROC'] + CLASS_NAMES
summary = summary.round(4)

print("\n" + "="*70)
print("InceptionTime — In-Distribution Results (diagonal of the 5x5 matrix)")
print("="*70)
print(summary.to_string(index=False))

summary_path = f'{PROCESSED_DIR}/inceptiontime_summary_table.csv'
summary.to_csv(summary_path, index=False)
print(f"\nSummary table saved to {summary_path}")

NameError: name 'CLASS_NAMES' is not defined